In [ ]:
from __future__ import annotations

import json
import re
from typing import Any


QUIZ_GENERATOR_VERSION = "2026-09-16-budget-v2"


class QuizGenerator:
    """
    Generates adaptive quizzes from project knowledge and
    persistent learner context.

    The generator does not access MongoDB directly.

    The service layer supplies:
    - retrieved project evidence
    - requested concepts
    - mastery state
    - recent assessment history
    - recent activity

    Groq is responsible for generating the questions.
    The generated structure is validated before being returned.
    """

    def __init__(self, ai_service):
        self.ai_service = ai_service

    # ========================================================
    # PUBLIC API
    # ========================================================

    def generate(
        self,
        evidence_chunks,
        concepts: list[str] | None = None,
        question_count: int = 5,
        difficulty: str = "medium",
        mastery: list[dict[str, Any]] | None = None,
        recent_performance: list[dict[str, Any]] | None = None,
        recent_activity: list[dict[str, Any]] | None = None,
        user_id: str | None = None,
        project_id: str | None = None,
    ) -> dict[str, Any]:

        if question_count < 1:
            raise ValueError(
                "question_count must be at least 1."
            )

        if question_count > 20:
            raise ValueError(
                "question_count cannot exceed 20."
            )

        if difficulty not in {
            "easy",
            "medium",
            "hard",
        }:
            raise ValueError(
                "difficulty must be easy, medium, or hard."
            )

        concepts = concepts or []
        mastery = mastery or []
        recent_performance = recent_performance or []
        recent_activity = recent_activity or []

        if not evidence_chunks:
            raise ValueError(
                "Cannot generate a quiz without project evidence."
            )

        prompt = self._build_prompt(
            evidence_chunks=evidence_chunks,
            concepts=concepts,
            question_count=question_count,
            difficulty=difficulty,
            mastery=mastery,
            recent_performance=recent_performance,
            recent_activity=recent_activity,
        )

        print(
            "[QuizGenerator] "
            f"evidence_chunks={self._evidence_chunk_count(prompt)} "
            f"approx_prompt_tokens={self._approx_tokens(prompt)} "
            f"question_count={question_count}"
        )

        raw_response = self.ai_service.generate_text(
            prompt=prompt,
            system_instruction=self._system_instruction(),
            operation="quiz_generation",
            user_id=user_id,
            project_id=project_id,
        )

        parsed = self._parse_response(
            raw_response
        )

        return self._validate_quiz(
            parsed=parsed,
            question_count=question_count,
            evidence_chunks=evidence_chunks,
            requested_concepts=concepts,
            difficulty=difficulty,
        )

    # ========================================================
    # PROMPT
    # ========================================================

    @staticmethod
    def _system_instruction() -> str:
        return """
You are an adaptive assessment generator for an AI learning platform.

Generate quiz questions ONLY from the supplied project evidence.

The learner context is provided to adapt:
- concept selection
- difficulty
- emphasis on weak areas
- recent mistakes
- recent performance

Never invent facts that are not supported by the project evidence.

Return ONLY valid JSON.

The JSON must have this structure:

{
  "questions": [
    {
      "id": "q1",
      "question": "...",
      "type": "mcq",
      "concept_id": "...",
      "difficulty": "easy|medium|hard",
      "options": ["...", "...", "...", "..."],
      "correct_option": "...",
      "expected_concepts": ["..."],
      "explanation": "...",
      "source_chunk_ids": ["..."]
    }
  ]
}

Rules:
- type must be "mcq" or "open_ended".
- MCQ questions must have exactly four options.
- correct_option must exactly match one option.
- open-ended questions must not require options.
- Every question must have a concept_id.
- Every question must have at least one source_chunk_id.
- Keep explanations concise.
- Do not include markdown fences.
"""

    @staticmethod
    def _approx_tokens(text: str) -> int:
        """Approximate tokens using a simple character-based estimate."""
        return (len(text) + 3) // 4

    @staticmethod
    def _evidence_chunk_count(prompt: str) -> int:
        """Count evidence objects in the generated prompt for diagnostics."""
        marker = 'PROJECT EVIDENCE:\n'
        if marker not in prompt:
            return 0
        try:
            evidence_text = prompt.split(marker, 1)[1]
            evidence_text = evidence_text.split('\n\nADAPTATION RULES:', 1)[0]
            return len(json.loads(evidence_text))
        except (ValueError, TypeError, json.JSONDecodeError):
            return 0

    def _build_prompt(
        self,
        evidence_chunks,
        concepts,
        question_count,
        difficulty,
        mastery,
        recent_performance,
        recent_activity,
    ) -> str:

        # Keep the combined user + system input comfortably below Groq's
        # 8k TPM limit. The system instruction is also sent to Groq, so the
        # user prompt gets a smaller budget than the total request.
        max_total_input_tokens = 6000
        system_tokens = self._approx_tokens(self._system_instruction())
        max_prompt_tokens = max(2500, max_total_input_tokens - system_tokens)
        # Hard character ceiling prevents accidentally crossing the legacy
        # 20,000-character application limit seen in older deployments.
        max_prompt_chars = 18000
        chunk_text_chars = 1400
        performance_limit = 5
        activity_limit = 8
        mastery_limit = 5
        context_chars = 6500

        def build(
            text_limit: int,
            performance_count: int,
            activity_count: int,
            mastery_count: int,
            context_limit: int,
        ) -> str:
            evidence = []

            for chunk in evidence_chunks[:10]:
                text = str(getattr(chunk, "text", "") or "")
                evidence.append(
                    {
                        "chunk_id": chunk.chunk_id,
                        "material_id": chunk.material_id,
                        "page_number": chunk.page_number,
                        "source_file_name": chunk.source_file_name,
                        "text": text[:text_limit],
                    }
                )

            weak_mastery = sorted(
                mastery,
                key=lambda item: float(
                    item.get("score", 0.0)
                ),
            )[:mastery_count]

            context = {
                "requested_concepts": concepts,
                "target_question_count": question_count,
                "requested_difficulty": difficulty,
                "weakest_concepts": weak_mastery,
                "recent_performance": recent_performance[:performance_count],
                "recent_activity": recent_activity[:activity_count],
            }

            context_json = json.dumps(
                context,
                default=str,
                indent=2,
            )

            if len(context_json) > context_limit:
                context["recent_activity"] = []
                context_json = json.dumps(
                    context,
                    default=str,
                    indent=2,
                )

            if len(context_json) > context_limit:
                context["recent_performance"] = []
                context_json = json.dumps(
                    context,
                    default=str,
                    indent=2,
                )

            if len(context_json) > context_limit:
                context["weakest_concepts"] = context["weakest_concepts"][:2]
                context_json = json.dumps(
                    context,
                    default=str,
                    indent=2,
                )

            return (
                "Create an adaptive quiz using the following "
                "project evidence and learner context.\n\n"
                "LEARNER CONTEXT:\n"
                f"{context_json}\n\n"
                "PROJECT EVIDENCE:\n"
                f"{json.dumps(evidence, default=str, indent=2)}\n\n"
                "ADAPTATION RULES:\n"
                "1. Prioritize requested concepts when supplied.\n"
                "2. Give additional attention to low-mastery concepts.\n"
                "3. Use recent performance to avoid repeatedly "
                "testing only concepts the learner already masters.\n"
                "4. Prefer concepts associated with recent weak "
                "performance or mistakes.\n"
                "5. Every question must be grounded in the supplied "
                "evidence.\n"
                "6. Use the requested difficulty as the target level.\n"
            )

        prompt = build(
            chunk_text_chars,
            performance_limit,
            activity_limit,
            mastery_limit,
            context_chars,
        )

        # Progressively reduce evidence/context until the request is
        # safely below the configured prompt budget. Metadata is retained
        # so source_chunk_ids and citations remain available for validation.
        reductions = [
            (1100, 4, 6, 4, 5000),
            (900, 3, 4, 3, 4000),
            (700, 2, 3, 2, 3000),
            (500, 1, 2, 1, 2200),
            (350, 1, 1, 1, 1600),
        ]

        for settings in reductions:
            if (
                self._approx_tokens(prompt) <= max_prompt_tokens
                and len(prompt) <= max_prompt_chars
            ):
                break
            prompt = build(*settings)

        if (
            self._approx_tokens(prompt) > max_prompt_tokens
            or len(prompt) > max_prompt_chars
        ):
            # Final deterministic evidence trim. Keep all evidence metadata
            # and only shorten the text fields further.
            prompt = build(
                250,
                1,
                1,
                1,
                1000,
            )

        if (
            self._approx_tokens(prompt) > max_prompt_tokens
            or len(prompt) > max_prompt_chars
        ):
            raise ValueError(
                "Quiz prompt exceeds the safe input budget after trimming."
            )

        return prompt

    # ========================================================
    # RESPONSE PARSING
    # ========================================================

    @staticmethod
    def _parse_response(
        raw_response: str,
    ) -> dict[str, Any]:

        text = raw_response.strip()

        # Remove accidental markdown fences.
        text = re.sub(
            r"^```(?:json)?\s*",
            "",
            text,
            flags=re.IGNORECASE,
        )

        text = re.sub(
            r"\s*```$",
            "",
            text,
        )

        try:
            parsed = json.loads(text)
        except json.JSONDecodeError as exc:

            # Try extracting the outer JSON object if the
            # model returned surrounding prose.
            match = re.search(
                r"\{.*\}",
                text,
                flags=re.DOTALL,
            )

            if not match:
                raise ValueError(
                    "Quiz generator returned invalid JSON."
                ) from exc

            try:
                parsed = json.loads(
                    match.group(0)
                )
            except json.JSONDecodeError as inner_exc:
                raise ValueError(
                    "Quiz generator returned invalid JSON."
                ) from inner_exc

        if not isinstance(parsed, dict):
            raise ValueError(
                "Quiz generator response must be a JSON object."
            )

        return parsed

    # ========================================================
    # VALIDATION
    # ========================================================

    def _validate_quiz(
        self,
        parsed: dict[str, Any],
        question_count: int,
        evidence_chunks,
        requested_concepts: list[str],
        difficulty: str,
    ) -> dict[str, Any]:

        questions = parsed.get(
            "questions"
        )

        if not isinstance(questions, list):
            raise ValueError(
                "Quiz response does not contain a questions list."
            )

        if len(questions) != question_count:
            raise ValueError(
                "Quiz generator returned "
                f"{len(questions)} questions; "
                f"expected {question_count}."
            )

        valid_chunk_ids = {
            chunk.chunk_id
            for chunk in evidence_chunks
        }

        validated_questions = []

        for index, question in enumerate(
            questions,
            start=1,
        ):

            if not isinstance(question, dict):
                raise ValueError(
                    f"Question {index} is not an object."
                )

            question_id = str(
                question.get(
                    "id",
                    f"q{index}",
                )
            )

            question_text = question.get(
                "question"
            )

            if not isinstance(
                question_text,
                str,
            ) or not question_text.strip():
                raise ValueError(
                    f"Question {index} has no valid question text."
                )

            question_type = question.get(
                "type"
            )

            if question_type not in {
                "mcq",
                "open_ended",
            }:
                raise ValueError(
                    f"Question {index} has invalid type."
                )

            concept_id = question.get(
                "concept_id"
            )

            if not isinstance(
                concept_id,
                str,
            ) or not concept_id.strip():
                raise ValueError(
                    f"Question {index} has no concept_id."
                )

            question_difficulty = question.get(
                "difficulty",
                difficulty,
            )

            if question_difficulty not in {
                "easy",
                "medium",
                "hard",
            }:
                raise ValueError(
                    f"Question {index} has invalid difficulty."
                )

            source_chunk_ids = question.get(
                "source_chunk_ids",
                [],
            )

            if not isinstance(
                source_chunk_ids,
                list,
            ) or not source_chunk_ids:
                raise ValueError(
                    f"Question {index} must have source_chunk_ids."
                )

            # Every citation must refer to evidence supplied
            # to the model.
            invalid_sources = [
                chunk_id
                for chunk_id in source_chunk_ids
                if chunk_id not in valid_chunk_ids
            ]

            if invalid_sources:
                raise ValueError(
                    f"Question {index} references unknown "
                    f"source chunks: {invalid_sources}"
                )

            options = question.get(
                "options",
                [],
            )

            correct_option = question.get(
                "correct_option"
            )

            if question_type == "mcq":

                if not isinstance(
                    options,
                    list,
                ) or len(options) != 4:
                    raise ValueError(
                        f"MCQ question {index} must have "
                        "exactly four options."
                    )

                options = [
                    str(option).strip()
                    for option in options
                ]

                if len(set(options)) != 4:
                    raise ValueError(
                        f"MCQ question {index} contains "
                        "duplicate options."
                    )

                if (
                    not isinstance(
                        correct_option,
                        str,
                    )
                    or correct_option not in options
                ):
                    raise ValueError(
                        f"MCQ question {index} has an invalid "
                        "correct_option."
                    )

            else:
                options = []
                correct_option = None

            expected_concepts = question.get(
                "expected_concepts",
                [],
            )

            if not isinstance(
                expected_concepts,
                list,
            ):
                expected_concepts = []

            explanation = question.get(
                "explanation"
            )

            if explanation is not None:
                explanation = str(
                    explanation
                ).strip()

            validated_questions.append(
                {
                    "id": question_id,
                    "question": question_text.strip(),
                    "type": question_type,
                    "concept_id": concept_id.strip(),
                    "difficulty": question_difficulty,
                    "options": options,
                    "correct_option": correct_option,
                    "expected_concepts": [
                        str(item)
                        for item in expected_concepts
                    ],
                    "explanation": explanation,
                    "source_chunk_ids": source_chunk_ids,
                }
            )

        return {
            "questions": validated_questions,
            "question_count": len(
                validated_questions
            ),
            "difficulty": difficulty,
            "adaptive": True,
        }